# FlyWire Research Framework: Missed Synapses Experiment

This notebook is the standard orchestration launcher for Missed Synapses biological experiments on Kaggle.
It serves strictly as a configuration and execution layer. All graph manipulation, preprocessing, and statistical logic remains enclosed within the core framework.

In [ ]:
# Cell 2: Imports (Framework Components Only)
import os
import zipfile
import pandas as pd
from pathlib import Path

from core.experiment_runner import ExperimentRunner, ExperimentConfig
from modules.error_models.error_registry import registry as error_registry
from modules.graph_analyses.analysis_registry import registry as analysis_registry
from modules.statistical_evaluation import StatisticalEvaluator
from core.export_manager import ExportManager


In [ ]:
# Cell 3: Runtime Configuration
# ==========================================
# EDIT THESE PARAMETERS TO RUN EXPERIMENTS
# ==========================================

EXPERIMENT = {
    "metadata": {
        "experiment_name": "Missed Synapses Validation",
        "author": "FlyWire Researcher",
        "description": "Assessing topological degradation from simulated missing synapses.",
        "notes": "Initial trial run."
    },
    "dataset": {
        "name": "TEST",
        "version": "v1"
    },
    "error": {
        "name": "missed_synapses",
        "rates": [0.00, 0.01, 0.05, 0.10, 0.20],
        "random_seeds": [1, 2, 3, 4, 5]
    },
    "biology": {
        "vulnerability_model": "degree_based_synaptic",
        "weights": {
            "synapse_weight": 1.0,
            "source_degree_weight": 0.5,
            "target_degree_weight": 0.5
        },
        "calibration": {}
    },
    "preprocessing": {
        "features": {
            "indegree": True,
            "outdegree": True,
            "pagerank": True,
            "hub_neighbor_count": True,
            "two_hop_size": True,
            "total_degree": False,
            "reciprocal_ratio": False
        }
    },
    "analysis": [
        "basic_structure",
        "degree_distribution",
        "pagerank",
        "connected_components",
        "reciprocity"
    ],
    "export": {
        "output_directory": "results",
        "create_zip": True,
        "save_intermediate_graphs": False,
        "save_statistics": True
    }
}


In [ ]:
# Cell 4: Locate & Extract Dataset
import os
import zipfile
from pathlib import Path

ds_name = EXPERIMENT['dataset']['name']
DATASET_ROOT = None

# Check for local demodata first (for local validation)
if os.path.exists('0-demodata'):
    for entry in os.listdir('0-demodata'):
        if entry.upper().startswith(ds_name.upper()):
            DATASET_ROOT = '0-demodata'
            print(f"Local demo dataset detected at {DATASET_ROOT}")
            break

# Fallback to Kaggle ZIP extraction
if not DATASET_ROOT:
    KAGGLE_INPUT_DIR = '/kaggle/input'
    KAGGLE_WORKING_DIR = '/kaggle/working/extracted_datasets'
    zip_files = []
    if os.path.exists(KAGGLE_INPUT_DIR):
        for root, _, files in os.walk(KAGGLE_INPUT_DIR):
            for file in files:
                if file.endswith('.zip'):
                    zip_files.append(os.path.join(root, file))
    
    if len(zip_files) == 0:
        raise FileNotFoundError("CRITICAL: No dataset ZIP files found in /kaggle/input and no local demodata found.")
    elif len(zip_files) > 1:
        print("Multiple ZIP files detected:")
        for z in zip_files:
            print(f" - {z}")
        raise ValueError("CRITICAL: Multiple ZIP files found. Please manually specify which archive to use or remove duplicates.")
    else:
        dataset_zip_path = zip_files[0]
        print(f"Dataset ZIP automatically detected: {dataset_zip_path}")
        print("Extracting archive... this may take a moment.")
        os.makedirs(KAGGLE_WORKING_DIR, exist_ok=True)
        with zipfile.ZipFile(dataset_zip_path, 'r') as zip_ref:
            zip_ref.extractall(KAGGLE_WORKING_DIR)
        print(f"Successfully extracted to {KAGGLE_WORKING_DIR}")
        DATASET_ROOT = KAGGLE_WORKING_DIR


In [ ]:
# Cell 5: Verify Dataset Structure
from core.dataset_registry import DatasetRegistry, DatasetRegistryError

try:
    registry = DatasetRegistry(configs_root='configs', dataset_root=DATASET_ROOT)
    resolved_dir = registry.resolve_dataset_dir(ds_name, DATASET_ROOT)
    print(f"Verified dataset structure. Resolved directory: {resolved_dir}")
except DatasetRegistryError as e:
    raise FileNotFoundError(f"CRITICAL: Could not resolve dataset folder for '{ds_name}' in '{DATASET_ROOT}'.") from e


In [ ]:
# Cell 6: Verify Registries
err_model = EXPERIMENT["error"]["name"]
print(f"Available Error Models: {error_registry.list_names()}")
print(f"Available Analyses: {analysis_registry.list_names()}")

if err_model not in error_registry.list_names():
    print(f"\nWARNING: '{err_model}' is not currently registered. Please ensure the scientific module is implemented in the framework before running.")


In [ ]:
# Cell 7: Instantiate Experiment Runner
runner = ExperimentRunner(analysis_registry, error_registry)
print("Experiment Runner instantiated successfully.")


In [ ]:
# Cell 8: Execute Experiments
results_per_rate = {}
base_out = Path(EXPERIMENT["export"]["output_directory"])
ds_name = EXPERIMENT["dataset"]["name"]
err_model = EXPERIMENT["error"]["name"]
seeds = EXPERIMENT["error"]["random_seeds"]

for err_rate in EXPERIMENT["error"]["rates"]:
    rate_str = f"{int(err_rate*100)}_percent"
    results_per_rate[err_rate] = []
    
    for trial, seed in enumerate(seeds, 1):
        print("\n======================================")
        print(f"Dataset      : {ds_name}")
        print(f"Error Model  : {err_model}")
        print(f"Error Rate   : {err_rate * 100:.1f}%")
        print(f"Trial        : {trial} / {len(seeds)}")
        print(f"Seed         : {seed}")
        print("======================================")
        
        trial_out = base_out / ds_name / err_model / rate_str / f"trial_{trial:03d}"
        
        config = ExperimentConfig(
            dataset_name=ds_name,
            dataset_root=str(DATASET_ROOT),
            error_model_name=err_model,
            error_model_config={
                "error_rate": err_rate,
                "biology": EXPERIMENT["biology"]
            },
            analysis_names=EXPERIMENT["analysis"],
            preprocessing_config={"features": EXPERIMENT["preprocessing"]["features"]},
            seed=seed,
            output_root=str(trial_out) if EXPERIMENT["export"]["save_statistics"] else None,
            create_zip=EXPERIMENT["export"]["create_zip"],
            extra={
                "metadata": EXPERIMENT["metadata"],
                "dataset_version": EXPERIMENT["dataset"]["version"],
                "save_intermediate_graphs": EXPERIMENT["export"]["save_intermediate_graphs"]
            }
        )
        
        res = runner.run(config)
        results_per_rate[err_rate].append(res)
        
        if res.succeeded:
            print(f"\n--> Success! Runtime: {res.runtime_seconds:.2f}s")
        else:
            print(f"\n--> Failed! Errors: {res.errors}")


In [ ]:
# Cell 9: Phase 017 - Statistical Evaluation & Phase 018 - Presentation Export
from core.checkpoint_manager import CheckpointManager
evaluator = StatisticalEvaluator()
aggregated_stats_by_rate = {}
baseline_runs = [r for r in results_per_rate.get(0.00, []) if r.succeeded]

for err_rate, run_results in results_per_rate.items():
    successful_runs = [r for r in run_results if r.succeeded]
    if successful_runs:
        eval_result = evaluator.evaluate(baseline_runs, successful_runs)
        aggregated_stats_by_rate[err_rate] = eval_result
        print(f"Evaluated stats for error rate {err_rate*100:.1f}% ({len(successful_runs)} successful trials).")
        
        cm = CheckpointManager(Path(EXPERIMENT['export']['output_directory']) / 'checkpoints')
        cm.save_phase_017_checkpoint(
            experiment_name=f"{EXPERIMENT['metadata']['experiment_name']}_{err_rate}",
            evaluation_result=eval_result,
            validation_results="VALIDATED"
        )
    else:
        print(f"No successful runs to evaluate for error rate {err_rate*100:.1f}%.")

# Phase 018: Generate Presentation Artifacts (Dashboards, Plots, ZIP)
ExportManager().export_presentation(
    results_by_rate=aggregated_stats_by_rate,
    output_root=Path(EXPERIMENT['export']['output_directory']),
    metadata=EXPERIMENT['metadata']
)
print("Phase 018 Presentation Export completed.")


In [ ]:
# Cell 10: Display Results
for err_rate, eval_result in aggregated_stats_by_rate.items():
    print(f"\n=================================================")
    print(f"Error Rate : {err_rate * 100:.1f}%")
    print("=================================================")
    
    for analysis_name, m_dict in eval_result.metrics.items():
        print(f"\nAnalysis : {analysis_name}")
        print("-" * 49)
        
        summary_data = []
        for m_name, ev in m_dict.items():
            summary_data.append({
                "Metric": m_name,
                "Baseline Mean": ev.baseline_mean,
                "Mean": ev.mean,
                "Std": ev.std,
                "CI Lower": ev.ci_lower,
                "CI Upper": ev.ci_upper,
                "Effect Size (d)": ev.effect_size
            })
        
        if summary_data:
            display(pd.DataFrame(summary_data))
        else:
            print("No statistics available for this analysis.")


In [ ]:
# Cell 11: Export Confirmation
print(f"\nAll intermediate structures and packages have been successfully exported to: {EXPERIMENT['export']['output_directory']}")
